## 하이퍼파라미터 기본 개념
하이퍼파라미터: 모델이 데이터에서 스스로 학습하는 값이 아니라, 사람이 학습 전에 미리 정하는 설정값임.

- alpha: 규제를 얼마나 강하게 적용할지 정하는 값임.
- l1_ratio: ElasticNet에서 L1 규제와 L2 규제를 어떤 비율로 섞을지 정하는 값임.

# Regularized Linear Models 규제 선형 회귀
 다항식이 복잡해지져서 회귀계수가 매우 크게 설정이되면서 과대적합이 되고 평가데이터세트에 대해서 형편없는 예측 성능을 보이게 된다.
비용함수의 최소값을 구하는 것이 모델이 추구하는 목적이므로, 비용함수의 최소값을 구하는데 𝝰 제약을 걸어 과적합을 방지하는 것을 규제라고 한다.
- Lasso L1방식의 규제 적용
- Ridge L2방식의 규제 적용
- ElasticNet L1, L2규제를 결합한 모델. 특성이 많은 데이터셋에 적용. L1규제로 특성개수를 줄이고, L2규제로 계수값의 크기도 조정할 수 있다.

**비용함수 목표**

$
\text{비용함수 목표} = \min \left(\text{RSS}(w) + \alpha \times W\right)
$

이 수식은 규제를 적용한 비용함수를 의미하며, 다음과 같은 요소들로 이루어져 있다:
1. **RSS(w)**: 이 부분은 Residual Sum of Squares의 약자로, 잔차 제곱합을 의미한다. 선형 회귀 모델에서 주로 사용하는 손실 함수로, 각 데이터 포인트에서의 예측 값과 실제 값 간의 차이를 제곱한 것들의 합이다. 즉, 모델의 예측 오차를 측정하는 부분이다. 수식으로는 다음과 같이 표현된다:
    $
    \text{RSS}(w) = \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2
    $
   여기서 $y_i$는 실제 값, $\hat{y}_i$는 모델의 예측 값이다.
2. **$\alpha$**: 규제 강도를 나타내는 하이퍼파라미터이다. 이 값이 클수록 규제의 효과가 커지고, 작을수록 규제의 효과가 줄어든다. 모델이 과적합되기 쉬운 경우, $\alpha$를 크게 설정하여 가중치를 제어할 수 있다.
3. **$W$**: 가중치들의 규제 항을 의미한다. 이는 가중치의 크기에 페널티를 부여하는 부분으로, 모델의 복잡도를 조절하는 역할을 한다.
   - L1 규제: $W = \sum_{j=1}^p |w_j|$
   - L2 규제: $W = \sum_{j=1}^p w_j^2$

수식을 다시 설명하면, 비용함수의 목표는 잔차 제곱합(RSS)과 규제 항($\alpha \times W$)을 더한 값을 최소화하는 것이다. 즉, 이 비용함수의 최적화 목표는 두 가지를 달성하고자 한다:
1. 모델의 예측 오차(RSS)를 줄이는 것.
2. 모델의 복잡도를 줄여서 가중치의 크기를 제어하는 것($\alpha \times W$).

**적합합 규제를 선택하려면 :**

1. **Lasso (L1 규제)**
  - 불필요한 피처를 자동으로 제거할 때 유용하다.
  - 많은 피처 중 일부만 중요할 때 사용하면 스파스한 모델을 만듦.
2. **Ridge (L2 규제)**
  - 모든 피처가 유의미하고 예측에 기여한다고 생각될 때 적합하다.
  - 피처가 많고 과적합을 방지하고 싶을 때 사용한다.
3. **Elastic Net (L1 + L2 규제)**
  - 피처를 일부 제거하면서도 나머지의 가중치도 줄이고 싶을 때 유용하다.
  - 상관관계가 높은 피처가 있을 때 선택을 안정적으로 한다.


## L2
- L2방식의 규제를 구현한 Ridge 클래스를 사용할 수 있다.
- 모든 피쳐의 회귀계수를 규제해 과적합을 방지한다.


## 실습 환경 준비

- NumPy: 배열 계산과 수치 연산을 다루기 위한 기본 라이브러리임.
- Pandas: 표 형태 데이터를 DataFrame으로 다루기 위한 라이브러리임.
- Matplotlib: 그래프를 그려 데이터 분포와 모델 결과를 시각화하는 라이브러리임.
- Seaborn: 통계 그래프를 더 쉽게 그리기 위한 시각화 라이브러리임.


In [1]:
# 초기 세팅용 import 구문입니다. 먼저 실행한 뒤 실습 코드를 작성합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyexpat import features
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet


## California Housing 데이터 불러오기

In [7]:
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing()

# 학습할 데이터
california_housing_df = pd.DataFrame(
    california_housing.data,
    columns=california_housing.feature_names,
)

# 정답 (y) - 예측해야 할 주택의 가격
california_housing_df['MedHouseValue'] = california_housing.target

california_housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseValue
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 학습/평가 데이터 분리

- train_test_split: 데이터를 학습용과 평가용으로 나누어 새 데이터 성능을 확인할 준비를 함.


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    california_housing.data,
    california_housing.target,
    test_size=0.2,
    random_state=42,
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(16512, 8) (16512,)
(4128, 8) (4128,)


## 회귀 평가지표

- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- MAE: 오차의 절댓값을 평균낸 값으로 실제 단위 해석이 쉬움.
- RMSE: MSE에 제곱근을 씌워 target과 같은 단위로 해석하는 지표임.
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.
- score: 모델의 기본 평가 점수를 계산하는 메서드임.


### 다항 feature - LinearRegression 모델 평가 지표

In [10]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# 모델 생성, 학습, 변환 작업을 순서
pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)

# 예측
model = pipeline.named_steps['model']

# 파이프라인 내에서 degree=2 증가 후 스케일링 -> 이를 학습한 LinearRegression을 가지고 예측
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

linear_eval_result = pd.DataFrame({
    'dataset': ['train', 'test'],
    'R2': [
        pipeline.score(X_train, y_train),
        pipeline.score(X_test, y_test)
    ],
    'MSE': [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_test, y_test_pred)
    ],
    'MAE': [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],
    'RMSE': [
        root_mean_squared_error(y_train, y_train_pred),
        root_mean_squared_error(y_test, y_test_pred)
    ]
})

print('회귀계수 제곱합:', np.sum(model.coef_ ** 2))
linear_eval_result

회귀계수 제곱합: 7170.956218226023


,dataset,R2,MSE,MAE,RMSE
0,train,0.685268,0.420727,0.460838,0.648634
1,test,0.645682,0.464302,0.467001,0.681397


## 다항 feature - Ridge L2 규제 선형 회귀 모델 평가 지표

In [11]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
# 모델 생성, 학습, 변환 작업을 순서
pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),

    # Ridge: 회귀 계수 제곱합을 줄이는 규제(penalty)
    ('model', Ridge(alpha=100))
])

pipeline.fit(X_train, y_train)

# 예측
model = pipeline.named_steps['model']

# 파이프라인 내에서 degree=2 증가 후 스케일링 -> 이를 학습한 Ridge을 가지고 예측
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

ridge_eval_result = pd.DataFrame({
    'dataset': ['train', 'test'],
    'R2': [
        pipeline.score(X_train, y_train),
        pipeline.score(X_test, y_test)
    ],
    'MSE': [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_test, y_test_pred)
    ],
    'MAE': [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],
    'RMSE': [
        root_mean_squared_error(y_train, y_train_pred),
        root_mean_squared_error(y_test, y_test_pred)
    ]
})

print('회귀계수 제곱합:', np.sum(model.coef_ ** 2))
ridge_eval_result

# L2 규제는 다항 feature의 표현력을 높이면서 alpha로 회귀계수를 제한
# alpha값에 따라서 R2, 오차지표 값이 달라진다.

회귀계수 제곱합: 2.442571268402181


,dataset,R2,MSE,MAE,RMSE
0,train,0.641622,0.479072,0.508727,0.692150
1,test,0.611686,0.508851,0.513345,0.713338


### 최적의 alpha값 찾기


In [14]:
from sklearn.model_selection import cross_val_score

# cross_val_score() : 학습데이터를 여러 fold로 나눠 검증 점수를 계산

alphas = [0, 0.1, 1, 10, 100, 200, 300]     # 테스트해볼 alpha 준비

cv_results = [] # 검증 점수를 모아둘 리스트

for alpha in alphas:
    pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=alpha))
    ])

    scores = -1 * cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,       # 데이터를 5개 fold(데이터 조각)
        scoring='neg_mean_squared_error'
    )

    cv_results.append({
        'alpha': alpha,
        'mean_MSE': scores.mean(),
        'std_MSE': scores.std(),
    })

ridge_cv_results = pd.DataFrame(cv_results)
ridge_cv_results

,alpha,mean_MSE,std_MSE
0,0.0,10.448255,18.601131
1,0.1,1.812390,2.335427
2,1.0,1.005593,0.782599
3,10.0,4.019982,6.748552
4,100.0,2.600972,4.182501
5,200.0,1.658760,2.318480
6,300.0,1.249920,1.498657


## 교차검증

- cross_val_score: 여러 fold의 검증 점수를 계산해 평균 성능을 더 안정적으로 확인함.


## L1
- L1규제방식을 구현한 Lasso클래스를 사용할 수 있다.
- alpha값을 통해 특정 회귀계수를 0까지 제한, 특정속성을 회귀계산에서 배제하는 것도 가능.


In [23]:
from sklearn.linear_model import Lasso

# alpha가 커질수록 강한 규제가 적용돼서 feature가 많이 제거될 수 있음
alphas = [0.002, 0.003, 0.07, 0.1, 0.5, 1]

# 비교 결과를 저장할 표, list 생성
coef_df = pd.DataFrame()
lasso_results = []

for alpha in alphas:
    pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', Lasso(alpha=alpha, max_iter=20000))
    ])

    pipeline.fit(X_train, y_train)

    lasso_model = pipeline.named_steps['model']
    feature_names = pipeline.named_steps['poly'].get_feature_names_out(california_housing.feature_names)

    # 각 alpha에서 핛브된 feature별 계수를 저장
    coef_df[f'alpha_{alpha}'] = pd.Series(
        lasso_model.coef_,
        index=feature_names
    )

    y_test_pred = pipeline.predict(X_test)

    lasso_results.append({
        'alpha': alpha,
        'MSE': mean_squared_error(y_test, y_test_pred),
        'R2': pipeline.score(X_train, y_train),
        'zero_coef_count': np.sum(np.isclose(lasso_model.coef_, 0))
    })

    # np.isclose(값, 0) : 값이 0에 가가우면 같은 것으로 판단

lasso_result = pd.DataFrame(lasso_results)
display(lasso_result)
display(coef_df)
# Lasso는 alpha가 커질수록 규제가 강해져 0이 되는 계수의 수가 증가한다.
# -> feature가 많이 제거되어 예측 성능이 하락할 수 있다.

,alpha,MSE,R2,zero_coef_count
0,0.002,0.498201,0.639045,19
1,0.003,0.505882,0.632814,20
2,0.070,0.659940,0.514706,41
3,0.100,0.669386,0.506707,41
4,0.500,0.937914,0.291651,43
5,1.000,1.310696,0.000000,44


,alpha_0.002,alpha_0.003,alpha_0.07,alpha_0.1,alpha_0.5,alpha_1
MedInc,0.791060,0.769874,0.000000,0.000000,0.00000,0.0
HouseAge,-0.000000,-0.000000,0.000000,0.000000,0.00000,0.0
AveRooms,-0.070148,-0.021389,-0.000000,-0.000000,0.00000,0.0
AveBedrms,0.000000,0.000000,0.000000,0.000000,0.00000,0.0
Population,-0.384126,-0.214918,0.000000,0.000000,0.00000,0.0
AveOccup,-0.446712,-0.244397,-0.000000,-0.000000,0.00000,0.0
Latitude,-0.788617,-0.795393,-0.000000,-0.000000,-0.00000,0.0
Longitude,-0.786894,-0.781340,-0.000000,-0.000000,0.00000,0.0
MedInc^2,-0.287064,-0.227106,0.000000,0.000000,0.00000,0.0
MedInc HouseAge,0.303042,0.268468,0.229670,0.212322,0.00000,0.0


## ElasticNet L1 + L2
L1규제, L2규제를 적절한 비율로 모두 적용하는 ElasticNet 선형회귀모델을 사용할 수 있다.
ElasticNet은 **회귀 분석** 기법 중 하나로, **Lasso**와 **Ridge**의 규제를 결합한 모델이다.
Lasso는 특성 선택에 효과적이고, Ridge는 모든 특성을 다루면서 모델을 규제한다.
ElasticNet은 이 두 가지 규제(L1과 L2)를 적절히 혼합하여 사용하는 방법이다.
**alpha 파라미터**
alpha는 a + b를 의미한다.
- a는 L1규제용 alpha값이다.
- b는 L2규제용 alpha값이다.
**l1_ratio 파라미터**
L1규제용 alpha값의 비율이다. $\frac{a}{a + b}$
- alpha가 10이고, l1_ratio가 0.7이면 a = 7, b = 3이다.
- alpha가 10이고, l1_ratio가 1이면 a = 10, b = 0이다. 즉, L1규제만 사용한다.
- alpha가 10이고, l1_ratio가 0이면 a = 0, b = 10이다. 즉, L2규제만 사용한다.
**수식:**
$$J(β) = RSS + α [ λ * ||β||₁ + (1 - λ) * ||β||₂² ]$$
- **RSS**: Residual Sum of Squares (예측 오차)
- **α**: 전체 규제 강도 (크면 규제가 강해짐)
- **λ**: L1과 L2 규제의 비율 조절 (0 ≤ λ ≤ 1)
  - λ = 1 → Lasso만 적용
  - λ = 0 → Ridge만 적용
- **||β||₁**: L1 노름 (∑|βᵢ|), 특성 선택
- **||β||₂²**: L2 노름 제곱 (∑βᵢ²), 계수 축소
**특징:**
- **Lasso와 Ridge의 장점을 결합**: ElasticNet은 Lasso의 **특성 선택** 능력과 Ridge의 **강한 규제** 특성을 모두 반영한다.
- **고차원 데이터에 적합**: 상관관계가 높은 특성이 많은 데이터나 차원이 높은 데이터에 적합하다.
- **Overfitting 방지**: 두 가지 규제를 혼합하여 과적합을 효과적으로 방지할 수 있다.


## 모델 학습

- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- score: 모델의 기본 평가 점수를 계산하는 메서드임.


In [24]:
from sklearn.linear_model import ElasticNet

pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),

    # ElasticNet : L1 + L2 규제를 함께 사용하는 모델
    # alpha: 전체 규제 정도
    # l1_ratio: L1의 규제 비율, 0에 가까울수록 Ridge, 1에 가까울수록 Lasso와 비슷하다
    # max_iter: 회귀계수의 최적값을 찾기 위한 반복 횟수
    ('model', ElasticNet(alpha=0.01, l1_ratio=0.1, max_iter=20000))
])

pipeline.fit(X_train, y_train)

elasticnet_result = pd.DataFrame({
    'dataset': ['train', 'test'],
    'R2' : [
        pipeline.score(X_train, y_train),
        pipeline.score(X_test, y_test)
    ]
})

elasticnet_result

,dataset,R2
0,train,0.634832
1,test,0.615396


## 다중공선성 MultiCollinearity
특성간의 상관관계가 너무 높은 경우를 가리킨다.
주택데이터에서 면적특성과 방의크기특성은 높은 상관관계(상관계수 0.8이상)를 가질수 있다.
다중공선성특성에 대한 회귀계수가 크게 학습이 되고, 이는 특정데이터에 민감한 과대적합을 유발한다.
**해결책**
- 다중공선성 특성 제거
- 규제모델을 사용한 회귀계수 억제


## 샘플 데이터 생성

In [25]:
np.random.seed(42)

# 첫 번째 feature X_corr를 만들고, 두 번째/세 번째 feature를 X_corr의 제곱/세제곱 기반으로 만듦.
# 이렇게 만들면 feature들이 서로 강하게 연관되어 다중공선성이 생김.
X_corr = np.random.rand(100, 1)
X_corr = np.column_stack((X_corr, X_corr ** 2 + 3, X_corr ** 3 + 4))

# target은 예제용 난수로 생성함.
y_corr = 3 * np.random.randn(100, 1) + np.random.randn(100, 1)
y_corr = y_corr.ravel()

# np.corrcoef(): 변수 간 상관계수를 계산함.
# rowvar=False: 행이 아니라 컬럼을 변수(feature)로 보고 상관계수를 계산함.
corr_mat = np.corrcoef(X_corr, rowvar=False)
corr_mat

array([[1.        , 0.96876011, 0.91658615],
       [0.96876011, 1.        , 0.98569849],
       [0.91658615, 0.98569849, 1.        ]])

## 학습/평가 데이터 분리

- train_test_split: 데이터를 학습용과 평가용으로 나누어 새 데이터 성능을 확인할 준비를 함.
- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.


In [26]:
# train_test_split(): 샘플 데이터를 학습용과 평가용으로 나눔.
X_corr_train, X_corr_test, y_corr_train, y_corr_test = train_test_split(
    X_corr,
    y_corr,
    test_size=0.2,
    random_state=42
)

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# 비교할 모델 목록임.
# LinearRegression: 규제 없음
# Ridge: L2 규제
# Lasso: L1 규제
# ElasticNet: L1 + L2 규제
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=10.0),
    'Lasso': Lasso(alpha=0.1, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
}

corr_results = []

for name, model in models.items():
    pipeline = Pipeline([
        # StandardScaler(): feature 스케일을 맞춰 규제 모델의 계수 벌점이 공정하게 적용되도록 함.
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    pipeline.fit(X_corr_train, y_corr_train)
    trained_model = pipeline.named_steps['model']

    corr_results.append({
        'model': name,
        'train_MSE': mean_squared_error(y_corr_train, pipeline.predict(X_corr_train)),
        'test_MSE': mean_squared_error(y_corr_test, pipeline.predict(X_corr_test)),
        'coef_squared_sum': np.sum(trained_model.coef_ ** 2),
        'zero_coef_count': np.sum(np.isclose(trained_model.coef_, 0))
    })

pd.DataFrame(corr_results)

# 회귀 계수 또는 feature 개수를 규제해서 과적합 해결이 가능함

,model,train_MSE,test_MSE,coef_squared_sum,zero_coef_count
0,LinearRegression,7.217016,10.176685,97.534503,0
1,Ridge,7.508614,10.527044,1.091929,0
2,Lasso,7.554207,10.601655,0.784001,1
3,ElasticNet,7.502225,10.505844,1.162947,1


In [27]:
np.random.seed(42)

n = 200

# x1은 핵심 feature
x1 = np.random.normal(0, 1, n)

# x2, x3는 x1과 거의 같은 feature
# 작은 noise만 더해서 거의 중복된 컬럼처럼 만듦.
x2 = x1 + np.random.normal(0, 0.01, n)
x3 = x1 + np.random.normal(0, 0.01, n)

# x4는 정답과 거의 관계없는 독립 feature
x4 = np.random.normal(0, 1, n)

X_corr = np.column_stack([x1, x2, x3, x4])
feature_names = ['x1', 'x2_almost_x1', 'x3_almost_x1', 'x4_noise']

# target은 실제로 x1의 영향을 받도록 만듦.
# 이렇게 해야 모델이 학습할 신호가 생김.
y_corr = 5 * x1 + np.random.normal(0, 0.5, n)

corr_df = pd.DataFrame(X_corr, columns=feature_names).corr()
corr_df

,x1,x2_almost_x1,x3_almost_x1,x4_noise
x1,1.000000,0.999944,0.999944,0.065398
x2_almost_x1,0.999944,1.000000,0.999886,0.064254
x3_almost_x1,0.999944,0.999886,1.000000,0.066621
x4_noise,0.065398,0.064254,0.066621,1.000000


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

X_corr_train, X_corr_test, y_corr_train, y_corr_test = train_test_split(
    X_corr,
    y_corr,
    test_size=0.2,
    random_state=42
)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=10.0),
    'Lasso': Lasso(alpha=0.05, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000),
}

results = []
coef_df = pd.DataFrame(index=feature_names)

for name, model in models.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    pipeline.fit(X_corr_train, y_corr_train)
    pred = pipeline.predict(X_corr_test)

    trained_model = pipeline.named_steps['model']
    coef_df[name] = trained_model.coef_

    results.append({
        'model': name,
        'test_R2': r2_score(y_corr_test, pred),
        'test_MSE': mean_squared_error(y_corr_test, pred),
        'coef_abs_sum': np.sum(np.abs(trained_model.coef_)),
        'zero_coef_count': np.sum(np.isclose(trained_model.coef_, 0))
    })

display(pd.DataFrame(results))
display(coef_df)

,model,test_R2,test_MSE,coef_abs_sum,zero_coef_count
0,LinearRegression,0.986293,0.258187,18.374808,0
1,Ridge,0.986690,0.250704,4.693612,0
2,Lasso,0.986987,0.245118,4.694631,1
3,ElasticNet,0.986773,0.249136,4.704208,0


,LinearRegression,Ridge,Lasso,ElasticNet
x1,9.968837,1.536908,4.645945,1.552183
x2_almost_x1,-6.798938,1.523699,0.000000,1.519741
x3_almost_x1,1.524100,1.536945,0.000686,1.559146
x4_noise,0.082932,0.096060,0.048000,0.073138
